# 02 — Feature Engineering: Molecular Descriptors

This notebook computes five sets of molecular descriptors from SMILES strings:
1. **Morgan Fingerprints** (ECFP4, 2048 bits) — circular topology
2. **RDKit Fingerprints** (2048 bits) — path-based
3. **MACCS Keys** (167 bits) — structural keys
4. **Physicochemical descriptors** (12 features) — MW, LogP, TPSA, etc.
5. **Combined** — Morgan FP + PhysChem (2060 features)

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors
from rdkit import DataStructs
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/processed/chembl_egfr_clean.csv")
print(f"Loaded {len(df)} compounds")

In [ ]:
def smiles_to_mol(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        return mol if mol is not None else None
    except:
        return None

# Convert SMILES to RDKit molecules
df["mol"] = df["canonical_smiles"].apply(smiles_to_mol)
valid = df["mol"].notna()
print(f"Valid molecules: {valid.sum()} / {len(df)}")
df = df[valid].copy()

In [ ]:
# 1. Morgan Fingerprints (ECFP4, radius=2, 2048 bits)
def morgan_fp(mol, radius=2, nbits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
    arr = np.zeros(nbits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X_morgan = np.vstack([morgan_fp(m) for m in df["mol"]])
print(f"Morgan FP: {X_morgan.shape}")

In [ ]:
# 2. RDKit Fingerprints (2048 bits)
def rdkit_fp(mol, nbits=2048):
    fp = Chem.RDKFingerprint(mol, fpSize=nbits)
    arr = np.zeros(nbits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X_rdkit = np.vstack([rdkit_fp(m) for m in df["mol"]])
print(f"RDKit FP: {X_rdkit.shape}")

In [ ]:
# 3. MACCS Keys (167 bits)
def maccs_fp(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros(167, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X_maccs = np.vstack([maccs_fp(m) for m in df["mol"]])
print(f"MACCS Keys: {X_maccs.shape}")

In [ ]:
# 4. Physicochemical descriptors
def physchem_desc(mol):
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        rdMolDescriptors.CalcTPSA(mol),
        Descriptors.NumRotatableBonds(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        Descriptors.FractionCSP3(mol),
        rdMolDescriptors.CalcNumHeavyAtoms(mol),
        Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
    ], dtype=np.float32)

X_physchem = np.vstack([physchem_desc(m) for m in df["mol"]])
# Replace NaN/Inf
X_physchem = np.nan_to_num(X_physchem, nan=0.0, posinf=0.0, neginf=0.0)
print(f"PhysChem: {X_physchem.shape}")

In [ ]:
# 5. Combined: Morgan + PhysChem
from sklearn.preprocessing import StandardScaler
X_physchem_scaled = StandardScaler().fit_transform(X_physchem)
X_combined = np.hstack([X_morgan, X_physchem_scaled])
print(f"Combined: {X_combined.shape}")

In [ ]:
# Save all descriptor sets
y_reg = df["pIC50"].values.astype(np.float32)
y_cls = (y_reg >= 6.0).astype(np.int32)

import os
os.makedirs("../data/processed", exist_ok=True)

for name, X in [("morgan", X_morgan), ("rdkit_fp", X_rdkit),
                ("maccs", X_maccs), ("physchem", X_physchem), ("combined", X_combined)]:
    np.savez_compressed(f"../data/processed/descriptors_{name}.npz",
                        X=X, y_reg=y_reg, y_cls=y_cls)
    print(f"Saved: descriptors_{name}.npz  shape={X.shape}")

In [ ]:
# Visualize descriptor statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Bit density for Morgan FP
bit_density = X_morgan.mean(axis=0)
axes[0].hist(bit_density, bins=50, color="#1565C0", alpha=0.7)
axes[0].set_xlabel("Bit density (fraction of compounds with bit=1)")
axes[0].set_ylabel("Number of bits")
axes[0].set_title("Morgan FP: Bit Density Distribution")

# PhysChem correlation heatmap
import seaborn as sns
physchem_names = ["MW", "LogP", "HBD", "HBA", "TPSA", "RotBonds",
                  "Rings", "ArRings", "CSP3", "HeavyAtoms", "MaxCharge", "MinCharge"]
df_physchem = pd.DataFrame(X_physchem, columns=physchem_names)
corr = df_physchem.corr()
sns.heatmap(corr, ax=axes[1], cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".1f", square=True, linewidths=0.5, annot_kws={"size": 7})
axes[1].set_title("PhysChem Descriptor Correlation")

# pIC50 vs LogP
sc = axes[2].scatter(X_physchem[:, 1], y_reg, c=y_cls, cmap="RdYlBu_r",
                     alpha=0.3, s=5, edgecolors="none")
plt.colorbar(sc, ax=axes[2], label="Active (1) / Inactive (0)")
axes[2].set_xlabel("LogP")
axes[2].set_ylabel("pIC50")
axes[2].set_title("pIC50 vs LogP")

plt.tight_layout()
plt.savefig("../results/plots/descriptor_analysis.png", dpi=150, bbox_inches="tight")
plt.show()